# Writing geopoints to a file

## Reading the input file

The encoder takes the data object we get from `from_source()`, a featurelist or a fieldlist. The file `metadata.gpt` holds one block with 44 points and a `#METADATA` section:

In [1]:
import numpy as np

import earthkit.data as ekd

ds = ekd.from_source("file", "metadata.gpt")
ds

path,metadata.gpt
size,1.6 KiB
types,"pandas, xarray, featurelist, fieldlist"


## Writing a file as it is

In [2]:
ds.to_target("file", "_res_copy.gpt")

with open("metadata.gpt", "rb") as f:
    original = f.read()

with open("_res_copy.gpt", "rb") as f:
    copy = f.read()

assert copy == original
print(f"the copy holds the same {len(copy)} bytes as the original")

the copy holds the same 1642 bytes as the original


## Writing a subset of the points

In [3]:
rows = ds.to_featurelist(mode="row")
rows[:3].to_target("file", "_res_rows.gpt")

The result has a single block, so it is a plain geopoints file and not a geopointset:

In [4]:
print(open("_res_rows.gpt").read())

#GEO
#METADATA
param=temperature
date=20130804
time=1200
level=0.2
#DATA
55.01 8.41 0.2 20130804 1200 294.4
54.33 8.6 0.2 20130804 1200 296.9
53.71 7.15 0.2 20130804 1200 297.1



Read back, the file holds the same number of points and the same values:

In [5]:
back = ekd.from_source("file", "_res_rows.gpt")
df = back.to_pandas()

assert len(df) == 3
np.testing.assert_array_equal(df["value"].to_numpy(), ds.to_pandas()["value"].to_numpy()[:3])
df

,latitude,longitude,level,date,value
0,55.01,8.41,0.2,2013-08-04 12:00:00,294.4
1,54.33,8.60,0.2,2013-08-04 12:00:00,296.9
2,53.71,7.15,0.2,2013-08-04 12:00:00,297.1


### Blocks of a geopointset

A `mode="block"` featurelist (the default) has one feature per block, so its subsets select blocks. The file `geopointset.gpts` has three blocks, only the second one has metadata:

In [6]:
s = ekd.from_source("file", "geopointset.gpts")
blocks = s.to_featurelist()
blocks.ls()

,param
0,None
1,geopotential
2,None


`sel()` selects the features by metadata. Written to a file, the block is a single block geopoints file, with its `#METADATA` section kept:

In [7]:
sub = blocks.sel(param="geopotential")
sub.to_target("file", "_res_sel.gpt")

print(open("_res_sel.gpt").read())

#GEO
#METADATA
param=geopotential
#DATA
60.82 23.5 0 20130512 600 101045.8



In [8]:
back = ekd.from_source("file", "_res_sel.gpt")
df = back.to_pandas()

assert len(back.to_featurelist()) == 1
assert len(df) == 1
assert df["value"].item() == 101045.8
assert df["latitude"].item() == 60.82
df

,latitude,longitude,level,date,value
0,60.82,23.5,0.0,2013-05-12 06:00:00,101045.8


More than one block makes a geopointset: the file starts with `#GEOPOINTSET` and every block keeps its own header and metadata:

In [9]:
blocks[:2].to_target("file", "_res_set.gpt")

print(open("_res_set.gpt").read())

#GEOPOINTSET
#GEO
#DATA
69.6523 18.9057 0 20130512 0 100869.8625
63.4882 10.8795 0 20130512 0 100282.3392
63.5657 10.694 0 20130512 0 100241.1666
61.2928 5.0443 0 20130512 0 99852.18932
#GEO
#METADATA
param=geopotential
#DATA
60.82 23.5 0 20130512 600 101045.8



In [10]:
back = ekd.from_source("file", "_res_set.gpt")

assert len(back.to_featurelist()) == 2
assert len(back.to_pandas()) == 5
np.testing.assert_array_equal(back.to_pandas()["value"].to_numpy(), blocks[:2].to_pandas()["value"].to_numpy())
back

path,_res_set.gpt
size,261
types,"pandas, xarray, featurelist, fieldlist"


## Writing with an explicit encoder

In [11]:
rows[:3].to_target("file", "_res_explicit.txt", encoder="geopoints")

assert open("_res_explicit.txt", "rb").read() == open("_res_rows.gpt", "rb").read()

In [12]:
encoder = ekd.create_encoder("geopoints", metadata={"author": "earthkit"})
blocks[1:2].to_target("file", "_res_meta.gpt", encoder=encoder)

print(open("_res_meta.gpt").read())

#GEO
#METADATA
param=geopotential
author=earthkit
#DATA
60.82 23.5 0 20130512 600 101045.8

